# 第 45 课｜SQLite 行情库与幂等增量

本 Notebook 对应「365 天量化金融」第 45 课。课程方法来自本地完整视频、课件截图和本地 Whisper 口播整理；这里把它重建为默认离线、可核对、可重复执行的量化教学实现。

- 图文口播：`/Users/mojiawen/Documents/学习资料归档/_暂存/pam-video-captures/365-days-quant-finance/45_youtube-7aCPO9jVHjI/index.html`
- 环境：`pkm-quant`
- 数据：离线模拟行情，写入本地教学数据库
- 状态：可运行学习稿；不代表策略收益承诺

> 网络接口和研究计算分离：Notebook 默认不请求第三方服务。真实接口只作为适配层示意，计算使用带来源声明的本地快照，避免限流、字段变化和历史数据重算破坏复现。

## 本节量化问题

`to_sql(if_exists="append")` 只会追加，不会自动去重。稳定行情库需要：

1. `PRIMARY KEY(symbol, date, adjustment)`；
2. 参数化查询和匹配查询模式的复合索引；
3. `UPSERT` 让重复同步幂等；
4. 事务保证一批更新要么全部成功、要么全部回滚。

In [1]:
from pathlib import Path
import json
import os
import platform
import sys
import tempfile
import timeit

# Matplotlib 首次运行需要写字体缓存；放到可写临时目录，避免污染用户目录。
MPL_CACHE_DIR = Path(tempfile.gettempdir()) / "pkm-quant-matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns

# 从本机字体中选择支持中文的字体，避免标题显示成方框。
preferred_chinese_fonts = [
    "PingFang SC",
    "Arial Unicode MS",
    "Hiragino Sans GB",
    "Heiti SC",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next(
    (name for name in preferred_chinese_fonts if name in available_font_names),
    "DejaVu Sans",
)
plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "实验" / "p2-python-toolchain",
        cwd / "p2-python-toolchain",
        *cwd.parents,
    ]
    for candidate in candidates:
        if (candidate / "课程辅助设计.md").exists():
            return candidate
    raise FileNotFoundError("请从 p2-python-toolchain 或量化交易项目目录启动 JupyterLab")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "fixtures"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 365

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("项目:", PROJECT_ROOT)
print("绘图字体:", chinese_font)

Python: 3.11.15
解释器: /opt/homebrew/Caskroom/miniforge/base/envs/pkm-quant/bin/python
项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain
绘图字体: PingFang SC


In [2]:
# sqlite3 属于 Python 标准库；数据库是可重复生成的实验产物。
import sqlite3

db_path = OUTPUT_DIR / "45_market_data.sqlite"
if db_path.exists():
    db_path.unlink()
connection = sqlite3.connect(db_path)
print("数据库:", db_path)

数据库: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain/outputs/45_market_data.sqlite


## 1. 先建约束，再写数据

唯一键包含复权口径，因为同一股票同一天的原始价、前复权价不是同一条记录。`CHECK` 约束在写入阶段阻止明显非法价格和成交量。

In [3]:
# 建立行情表和“股票 + 日期”查询索引。
connection.executescript("""
CREATE TABLE prices (
    symbol TEXT NOT NULL,
    date TEXT NOT NULL,
    adjustment TEXT NOT NULL,
    open REAL NOT NULL CHECK(open > 0),
    high REAL NOT NULL CHECK(high > 0),
    low REAL NOT NULL CHECK(low > 0),
    close REAL NOT NULL CHECK(close > 0),
    volume INTEGER NOT NULL CHECK(volume >= 0),
    source TEXT NOT NULL,
    PRIMARY KEY (symbol, date, adjustment)
);
CREATE INDEX idx_prices_symbol_date ON prices(symbol, date);
""")
schema = pd.read_sql_query("PRAGMA table_info(prices)", connection)
display(schema[["name", "type", "notnull", "pk"]])
assert {"symbol", "date", "adjustment", "close"}.issubset(schema["name"])

,name,type,notnull,pk
0,symbol,TEXT,1,1
1,date,TEXT,1,2
2,adjustment,TEXT,1,3
3,open,REAL,1,0
4,high,REAL,1,0
5,low,REAL,1,0
6,close,REAL,1,0
7,volume,INTEGER,1,0
8,source,TEXT,1,0


## 2. UPSERT 让重复同步不重复插入

`ON CONFLICT ... DO UPDATE` 以唯一键定位已有行。下面把同一批数据写两次，行数仍保持不变；这就是本实验的幂等性证据。

In [4]:
# 准备 12 行标准记录，并定义参数化 UPSERT。
daily = pd.read_csv(DATA_DIR / "daily_prices.csv", parse_dates=["date"])
batch = (
    daily.query("symbol == 'BANK_A'")
    .head(12)
    .assign(
        date=lambda x: x["date"].dt.strftime("%Y-%m-%d"),
        adjustment="none",
        source="fixture",
    )
)
columns = ["symbol", "date", "adjustment", "open", "high", "low", "close", "volume", "source"]
sql = """
INSERT INTO prices(symbol,date,adjustment,open,high,low,close,volume,source)
VALUES(?,?,?,?,?,?,?,?,?)
ON CONFLICT(symbol,date,adjustment) DO UPDATE SET
  open=excluded.open, high=excluded.high, low=excluded.low,
  close=excluded.close, volume=excluded.volume, source=excluded.source
"""
records = list(batch[columns].itertuples(index=False, name=None))
with connection:
    connection.executemany(sql, records)
with connection:
    connection.executemany(sql, records)

row_count = connection.execute("SELECT COUNT(*) FROM prices").fetchone()[0]
print("重复写入两次后的行数:", row_count)
assert row_count == len(batch) == 12

重复写入两次后的行数: 12


## 3. 参数化查询与查询计划

问号 `?` 是参数占位符，值不会被拼进 SQL 文本。`EXPLAIN QUERY PLAN` 用于确认“股票 + 日期区间”查询命中索引，而不是凭小数据耗时猜测。

In [5]:
# 按代码和日期区间查询，并检查 SQLite 的执行计划。
query = """
SELECT symbol,date,close,volume
FROM prices
WHERE symbol=? AND date>=? AND date<=?
ORDER BY date
"""
params = ("BANK_A", "2023-01-05", "2023-01-13")
result = pd.read_sql_query(query, connection, params=params, parse_dates=["date"])
plan = connection.execute("EXPLAIN QUERY PLAN " + query, params).fetchall()

display(result)
display(pd.DataFrame(plan, columns=["id", "parent", "unused", "detail"]))
assert result["date"].is_monotonic_increasing
assert any("idx_prices_symbol_date" in row[-1] for row in plan)

,symbol,date,close,volume
0,BANK_A,2023-01-05,11.6864,1974821
1,BANK_A,2023-01-06,11.6461,3999555
2,BANK_A,2023-01-09,11.4919,2813450
3,BANK_A,2023-01-10,11.4810,5359349
4,BANK_A,2023-01-11,11.4141,3812403
5,BANK_A,2023-01-12,11.4707,2180490
6,BANK_A,2023-01-13,11.5004,3492340


,id,parent,unused,detail
0,4,0,49,SEARCH prices USING INDEX idx_prices_symbol_da...


## 4. 事务失败必须回滚整批

第二行故意写入负价格，触发 `CHECK`。由于两行在同一个 `with connection:` 事务内，前一行也不会残留。

In [6]:
# 捕获教学性完整性错误，并验证原子回滚。
before = connection.execute("SELECT COUNT(*) FROM prices").fetchone()[0]
bad_records = [
    ("DEMO_A", "2025-01-02", "none", 10, 11, 9, 10.5, 100, "demo"),
    ("DEMO_A", "2025-01-03", "none", 10, 11, 9, -1.0, 100, "demo"),
]
try:
    with connection:
        connection.executemany(sql, bad_records)
except sqlite3.IntegrityError as exc:
    print("已捕获并回滚:", type(exc).__name__)

after = connection.execute("SELECT COUNT(*) FROM prices").fetchone()[0]
print("事务前/后行数:", before, after)
assert before == after
connection.close()

已捕获并回滚: IntegrityError
事务前/后行数: 12 12


## 方法边界与复用结论


- SQLite 适合单机、读多写少的研究库；高并发写入需换更合适的数据库。
- 索引提高读性能但占空间并降低写速，应按真实查询模式建立。
- 数据库文件仍需备份、版本和来源清单；“存进数据库”不等于数据可信。


本课代码验证的是方法和数据契约，不是未来收益。迁移到真实研究前，还需重新核对来源授权、复权、时区、可得时间、交易成本和样本外边界。